# Quantization for Inference and Training

| Difficulty | ██████░░░░ 6/10 |
| :--- | :--- |
| Prerequisites | 06.01, 06.06 |
| Optional | 05.01 (entropy — for rate-distortion connection), 08.06 (LoRA — for QLoRA) |
| Time | ~75 min |
| Colab GPU | Not needed |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NiravRVaghasiya/maths-for-ai/blob/main/06_Numerical_Methods/08_quantization.ipynb)

---

## 🎯 Learning Objective

Understand the mathematics of mapping floating-point weights to low-bit integers — the core
technique behind every deployed LLM — and implement post-training quantization (RTN, GPTQ,
AWQ), quantization-aware training (STE), and NF4 (QLoRA) from scratch.

---

## 📐 Theory

`06.06` covered *training*-time mixed precision: keep master weights in float32, compute in
float16/bfloat16, and dynamically scale the loss to avoid underflow. This notebook covers
*inference*-time (and training-time) **quantization**: representing weights (and sometimes
activations) in even lower precision — typically 8-bit or 4-bit integers — to slash memory
and compute costs at deployment.

### 1. The quantization problem

Map a continuous-valued weight $w \in \mathbb{R}$ (stored as float16/32) to a discrete value
in a finite set $\{q_0, q_1, \ldots, q_{2^b - 1}\}$ representable in $b$ bits. This is a
**rate-distortion** tradeoff (`05.01`): fewer bits $=$ less storage/compute, but more
distortion (quantization error).

**Uniform (linear) quantization** maps a floating-point weight to a $b$-bit integer and back:

$$q(w) = \text{clamp}\!\left(\left\lfloor \frac{w - z}{s} \right\rceil,\; 0,\; 2^b - 1\right), \qquad \hat{w} = s \cdot q(w) + z$$

where $s$ is the **scale** (step size) and $z$ is the **zero point** (offset):

$$s = \frac{w_{\max} - w_{\min}}{2^b - 1}, \qquad z = w_{\min}$$

**Symmetric quantization** ($z = 0$): simpler, standard for weights. The scale is
$s = \max(|w|) \,/\, (2^{b-1} - 1)$, and the integer range is $[-(2^{b-1}-1),\; 2^{b-1}-1]$.

**Asymmetric quantization** ($z \neq 0$): better for activations with non-symmetric
distributions (e.g., post-ReLU, which is always $\geq 0$). Uses the full $[0, 2^b - 1]$
range.

### 2. Quantization error analysis

For uniform quantization with step size $s$, the worst-case error on any single weight is:

$$|w - \hat{w}| \leq \frac{s}{2} = \frac{w_{\max} - w_{\min}}{2^{b+1} - 2}$$

The **mean squared quantization error** for a value uniformly distributed within one bin:
$\text{MSE} = s^2 / 12$ — the same formula as quantization noise in signal processing.

**Per-tensor vs. per-channel vs. group quantization.** Applying one $(s, z)$ pair per entire
weight matrix (per-tensor) is the coarsest granularity: outlier weights in any single channel
inflate the range for every channel, wasting precious bits on channels with much smaller
dynamic range. Per-output-channel quantization assigns a separate $(s, z)$ per row of the
weight matrix — much better, since each output channel has its own range.

**Group quantization** (the modern standard) uses one $(s, z)$ per group of $g$ contiguous
weights (typically $g = 128$). This is the granularity used by GPTQ, AWQ, and GGUF. The
tradeoff: smaller groups $\rightarrow$ less quantization error but more overhead (each group
stores its own scale/zero-point, typically in float16, adding $2 \times 16 / g$ bits per
weight of metadata overhead).

### 3. Post-training quantization (PTQ)

Quantize a pretrained model *without* retraining. The challenge: choose $(s, z)$ to minimize
*output* degradation, not just per-weight error.

**Round-to-nearest (RTN):** The simplest approach — quantize each weight independently to its
nearest representable value. Works acceptably for INT8, but often fails for INT4: too much
information is destroyed per weight.

**GPTQ (Frantar et al., 2022):** Quantize weights one column at a time, using the **inverse
Hessian** (approximated via the calibration data's $\mathbf{X}^T\mathbf{X}$) to optimally
redistribute the rounding error from each quantized weight to the remaining un-quantized
weights. For each column $j$ of the weight matrix $\mathbf{W}$:

1. Quantize $w_j \rightarrow \hat{w}_j$ (round to nearest).
2. Compute the rounding error $\delta_j = w_j - \hat{w}_j$.
3. Update the remaining (un-quantized) weights: $\mathbf{w}_{j+1:} \leftarrow \mathbf{w}_{j+1:} - \delta_j \cdot \frac{[\mathbf{H}^{-1}]_{j, j+1:}}{[\mathbf{H}^{-1}]_{jj}}$.

This minimizes $\|\mathbf{W}\mathbf{x} - \hat{\mathbf{W}}\mathbf{x}\|^2$ by an
**optimal brain compression** (OBC) argument: each column's quantization error is
compensated by adjusting later columns in the direction that the Hessian says hurts the
output least.

**AWQ (Lin et al., 2023):** Not all weights matter equally — a small fraction of "salient"
channels (those multiplied by large activations) dominate model quality. AWQ protects salient
weights by **scaling them up** before quantization (and compensating with the inverse scale on
activations):

$$\hat{\mathbf{W}} = \text{quant}(\mathbf{W} \cdot \text{diag}(\mathbf{s})) \cdot \text{diag}(\mathbf{s})^{-1}$$

The per-channel scale $s_j$ is chosen to minimize quantization MSE weighted by activation
magnitude: $s_j^* = \arg\min_{s_j} \mathbb{E}\bigl[\|\mathbf{W}\mathbf{x} - \hat{\mathbf{W}}\mathbf{x}\|^2\bigr]$.
Larger $s_j$ reduces relative quantization error on channel $j$ (its effective precision
increases) but inflates the dynamic range; the optimal balance depends on $\mathbb{E}[x_j^2]$.

### 4. Quantization-aware training (QAT)

Insert **fake-quantize** operations (quantize then immediately dequantize) into the forward
pass during training, so the model learns to be robust to quantization noise:

$$\hat{w} = s \cdot \text{clamp}\!\left(\left\lfloor \frac{w}{s} \right\rceil,\; -2^{b-1}+1,\; 2^{b-1}-1\right)$$

The gradient through the rounding step uses the **straight-through estimator (STE):**
pretend the rounding function has gradient 1 (its true gradient is 0 almost everywhere, and
undefined at integers):

$$\frac{\partial \hat{w}}{\partial w} \approx \begin{cases} 1 & \text{if } w \in [w_{\min}, w_{\max}] \\ 0 & \text{otherwise (clipped)} \end{cases} \qquad \text{(STE)}$$

This is mathematically unjustified — the rounding function is piecewise constant, so its
true derivative is 0 — but empirically effective. The model learns weights that, when
quantized, produce outputs close to what the full-precision model would have produced.

### 5. QLoRA and NF4

**QLoRA (Dettmers et al., 2023)** quantizes the base model to **NF4** (4-bit NormalFloat),
then applies LoRA (`08.06`) low-rank adapters on top. The combination gives near-full-
precision fine-tuning quality at $\sim\!4\times$ memory reduction.

The mathematical insight behind NF4: neural network weights are approximately
$\mathcal{N}(0, \sigma^2)$-distributed. For Gaussian data, the **information-theoretically
optimal** quantization levels are *not* equally spaced — they have **equal probability
mass** in each bin, not equal width. Since the Gaussian density is highest near 0, the
levels cluster more densely near 0 (where most weights live) and spread out in the tails
(where few weights live). Concretely, the 16 NF4 levels are the centroids of equal-
probability bins of $\mathcal{N}(0, 1)$ — computed once analytically via the inverse CDF
$\Phi^{-1}$:

$$\text{NF4 levels: } \bigl\{\mathbb{E}[X \mid X \in [\Phi^{-1}(k/16),\; \Phi^{-1}((k+1)/16)]]\bigr\}_{k=0}^{15}$$

NF4 also uses **double quantization**: the per-group scale factors (themselves float16) are
quantized a second time to 8-bit, reducing overhead to $\sim\!0.5$ bits/weight.

### 6. The quantization-precision-perplexity tradeoff

Empirically, quantizing most LLMs to:

- **INT8**: $<1\%$ perplexity increase. Essentially lossless for inference.
- **INT4** (with GPTQ/AWQ + group quantization, $g=128$): $<1\%$ perplexity increase. The
  current sweet spot for LLM deployment — $4\times$ memory reduction from float16.
- **INT4** (naive RTN, no calibration): $1$–$5\%$ perplexity increase.
- **INT3**: Significant degradation for most models. Below 4 bits, quality degrades rapidly
  — this is the empirical "cliff" in the bit-width vs. quality curve.

This defines the practical deployment frontier: **INT4 with group quantization is the
current sweet spot** for LLM inference. A 70B-parameter model in float16 needs 140 GB;
in INT4, it fits in $\sim\!35$ GB — the difference between needing a server and fitting
on a single consumer GPU.

---

In [ ]:
# Setup
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy import stats
import matplotlib.pyplot as plt
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

## 👁️ Visual Intuition

Three visualizations that build intuition for why quantization works — and where it breaks:

1. **Weight distribution vs. quantization grid:** Overlaying INT8 and INT4 grid lines on a
   realistic weight histogram shows how coarser grids force many distinct weights into the
   same bin, and how outliers waste grid resolution on sparsely-populated regions.
2. **Perplexity vs. bit-width:** The characteristic "cliff" below 4 bits — a gentle slope from
   32 to 4 bits, then a steep dropoff.
3. **Per-channel distributions:** Different output channels have wildly different dynamic
   ranges, explaining why per-channel quantization dramatically outperforms per-tensor.

In [ ]:
torch.manual_seed(42)
np.random.seed(42)

# --- Simulate realistic weight distribution (mixture of Gaussians + outliers) ---
weights = torch.cat([
    torch.randn(9500) * 0.02,       # bulk of weights: tight Gaussian
    torch.randn(500) * 0.15,         # heavier-tailed channels
])
# Inject a few outliers (as observed in real LLMs)
weights[0], weights[1] = 0.45, -0.38

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

# --- Panel 1: Weight histogram with quantization grids ---
ax = axes[0]
ax.hist(weights.numpy(), bins=120, density=True, alpha=0.7, color="#4C72B0", edgecolor="none")
w_min, w_max = weights.min().item(), weights.max().item()
for bits, color, label in [(8, "#55A868", "INT8 grid (256 levels)"),
                            (4, "#C44E52", "INT4 grid (16 levels)")]:
    n_levels = 2**bits
    grid = np.linspace(w_min, w_max, n_levels + 1)
    for g in grid:
        ax.axvline(g, color=color, alpha=0.15 if bits == 8 else 0.4, lw=0.5 if bits == 8 else 1.2)
    ax.axvline(grid[0], color=color, alpha=0.6, lw=1.5, label=label)  # legend entry
ax.set_xlabel("weight value"); ax.set_ylabel("density")
ax.set_title("Weight distribution vs. quantization grids\nOutliers inflate the range, wasting bits")
ax.legend(fontsize=8); ax.set_xlim(w_min - 0.02, w_max + 0.02)

# --- Panel 2: Perplexity vs. bit-width (simulated characteristic curve) ---
ax = axes[1]
bit_widths = np.array([16, 8, 6, 5, 4, 3, 2])
# Characteristic curve: near-flat 16->4, cliff below 4
ppl_gptq = np.array([5.68, 5.70, 5.72, 5.78, 5.85, 7.20, 15.5])  # with calibration
ppl_rtn  = np.array([5.68, 5.72, 5.82, 6.10, 6.55, 9.80, 28.0])  # naive RTN
ax.plot(bit_widths, ppl_gptq, 'o-', color="#55A868", lw=2, markersize=7, label="GPTQ (calibrated)")
ax.plot(bit_widths, ppl_rtn,  's--', color="#C44E52", lw=2, markersize=7, label="RTN (naive)")
ax.axvspan(3.5, 4.5, alpha=0.12, color="#55A868", label="sweet spot (INT4)")
ax.set_xlabel("bit-width"); ax.set_ylabel("perplexity")
ax.set_title('Perplexity vs. bit-width\n"Cliff" below 4 bits')
ax.set_xticks(bit_widths); ax.invert_xaxis()
ax.legend(fontsize=8)

# --- Panel 3: Per-channel distributions ---
ax = axes[2]
torch.manual_seed(7)
# Simulate a weight matrix with channels of very different scales
W = torch.randn(4, 256)
W[0] *= 0.01   # narrow channel
W[1] *= 0.05
W[2] *= 0.15   # wide channel
W[3] *= 0.40   # very wide channel (has outliers)
colors = ["#4C72B0", "#55A868", "#DD8452", "#C44E52"]
for i in range(4):
    ax.hist(W[i].numpy(), bins=50, density=True, alpha=0.55, color=colors[i],
            label=f"ch {i} (\u03c3={W[i].std():.3f})", edgecolor="none")
# Show per-tensor scale (one grid for all)
global_max = W.abs().max().item()
for g in np.linspace(-global_max, global_max, 17):
    ax.axvline(g, color="gray", alpha=0.25, lw=0.8)
ax.set_xlabel("weight value"); ax.set_ylabel("density")
ax.set_title("Per-channel distributions\nOne global scale wastes bits on narrow channels")
ax.legend(fontsize=7)

plt.tight_layout()
plt.show()

# Quantitative: per-tensor vs per-channel MSE
def quantize_symmetric(w, bits):
    qmax = 2**(bits-1) - 1
    s = w.abs().max() / qmax
    q = torch.clamp(torch.round(w / s), -qmax, qmax)
    return q * s

per_tensor_mse = ((W - quantize_symmetric(W, 4))**2).mean().item()
per_channel_mse = sum(((W[i] - quantize_symmetric(W[i], 4))**2).mean().item() for i in range(4)) / 4
print(f"INT4 quantization MSE  --  per-tensor: {per_tensor_mse:.6f}  vs  per-channel: {per_channel_mse:.6f}")
print(f"Per-channel is {per_tensor_mse/per_channel_mse:.1f}x better (each channel uses its own scale).")

## 🐍 Implementation from Scratch

We implement five core quantization primitives, each building on the math from the Theory
section:

1. **Symmetric quantization** (absmax scaling, zero point = 0)
2. **Asymmetric quantization** (min-max scaling, nonzero zero point)
3. **Group quantization** (one scale/zero-point per $g$ contiguous weights)
4. **Straight-through estimator (STE) fake-quantize** for QAT
5. **NF4 optimal quantization levels** for Gaussian-distributed weights

In [ ]:
# --- 1. Symmetric quantization (zero point = 0) ---
def quantize_symmetric(w, bits=8):
    """Symmetric uniform quantization: maps w to b-bit signed integers, then dequantizes."""
    qmax = 2**(bits - 1) - 1  # e.g. 127 for INT8
    scale = w.abs().max() / qmax
    scale = max(scale.item(), 1e-10)
    q_int = torch.clamp(torch.round(w / scale), -qmax, qmax)
    w_hat = q_int * scale  # dequantize
    return w_hat, scale, q_int

# Demo
w_demo = torch.tensor([0.23, -1.05, 0.78, -0.12, 2.41])
for bits in [8, 4]:
    w_hat, scale, q_int = quantize_symmetric(w_demo, bits)
    mse = ((w_demo - w_hat)**2).mean().item()
    print(f"INT{bits}: scale={scale:.5f}, int_vals={q_int.tolist()}, MSE={mse:.6f}")
    print(f"  original:       {w_demo.tolist()}")
    print(f"  reconstructed:  {w_hat.tolist()}\n")

# --- 2. Asymmetric quantization (nonzero zero point) ---
def quantize_asymmetric(w, bits=8):
    """Asymmetric uniform quantization: uses the full [0, 2^b-1] unsigned range."""
    qmin, qmax = 0, 2**bits - 1
    w_min, w_max = w.min(), w.max()
    scale = (w_max - w_min) / (qmax - qmin)
    scale = max(scale.item(), 1e-10)
    zero_point = round((-w_min.item()) / scale)
    zero_point = max(qmin, min(qmax, zero_point))
    q_int = torch.clamp(torch.round(w / scale + zero_point), qmin, qmax)
    w_hat = (q_int - zero_point) * scale
    return w_hat, scale, zero_point, q_int

# Demo: post-ReLU activations (all >= 0)
activations = torch.tensor([0.0, 0.15, 0.82, 0.03, 1.47, 0.0, 0.56])
w_hat_asym, s_asym, zp, q_asym = quantize_asymmetric(activations, 8)
print(f"Asymmetric INT8 on post-ReLU: scale={s_asym:.5f}, zero_point={zp}")
print(f"  MSE = {((activations - w_hat_asym)**2).mean().item():.8f}\n")

# --- 3. Group quantization (one scale per g contiguous weights) ---
def quantize_group(w, bits=4, group_size=128):
    """Group quantization: split w into groups of `group_size`, quantize each independently."""
    orig_shape = w.shape
    w_flat = w.reshape(-1)
    # Pad to multiple of group_size
    pad_len = (group_size - len(w_flat) % group_size) % group_size
    w_padded = F.pad(w_flat, (0, pad_len))
    w_groups = w_padded.reshape(-1, group_size)

    qmax = 2**(bits - 1) - 1
    scales = w_groups.abs().amax(dim=1, keepdim=True) / qmax  # one scale per group
    scales = scales.clamp(min=1e-10)
    q_int = torch.clamp(torch.round(w_groups / scales), -qmax, qmax)
    w_hat_groups = q_int * scales
    total = len(w_flat)
    w_hat = w_hat_groups.reshape(-1)[:total]
    return w_hat.reshape(orig_shape), scales.squeeze()

# Demo: group quantization on a 1024-element vector
torch.manual_seed(0)
w_large = torch.randn(1024) * 0.05
w_large[500] = 1.5  # inject one outlier
print("Group quantization INT4 on 1024 weights (one outlier injected):")
for g in [32, 64, 128, 1024]:
    w_hat_g, _ = quantize_group(w_large, bits=4, group_size=g)
    mse_g = ((w_large - w_hat_g)**2).mean().item()
    n_scales = (1024 + g - 1) // g
    print(f"  group_size={g:4d}: MSE={mse_g:.6f}, #scales={n_scales}")
print("  Smaller groups -> less error, more overhead (more stored scales)\n")

# --- 4. Straight-Through Estimator (STE) fake-quantize ---
class FakeQuantize(torch.autograd.Function):
    """Fake quantize: quantize in forward, straight-through in backward (gradient ~ 1)."""
    @staticmethod
    def forward(ctx, w, bits=8):
        qmax = 2**(bits - 1) - 1
        scale = w.abs().max() / qmax
        scale = max(scale.item(), 1e-10)
        q = torch.clamp(torch.round(w / scale), -qmax, qmax)
        w_hat = q * scale
        # Save mask for STE: gradient is 1 inside clamp range, 0 outside
        bounds = torch.tensor([-qmax * scale, qmax * scale])
        ctx.save_for_backward(w, bounds)
        return w_hat

    @staticmethod
    def backward(ctx, grad_output):
        w, bounds = ctx.saved_tensors
        # STE: pass gradient through where w is within representable range
        mask = (w >= bounds[0]) & (w <= bounds[1])
        return grad_output * mask.float(), None  # None for bits

fake_quantize = FakeQuantize.apply

# Demo: gradient flows through fake-quantize
w_qat = torch.randn(5, requires_grad=True)
w_hat_qat = fake_quantize(w_qat, 4)
loss_qat = (w_hat_qat ** 2).sum()
loss_qat.backward()
print("STE fake-quantize gradient check:")
print(f"  w         = {[round(v, 4) for v in w_qat.data.tolist()]}")
print(f"  w_hat     = {[round(v, 4) for v in w_hat_qat.data.tolist()]}")
print(f"  dL/dw STE = {[round(v, 4) for v in w_qat.grad.tolist()]}")
print("  (Rounding has gradient 0, but STE pretends it is 1 -> gradients flow through)\n")

# --- 5. NF4: information-theoretically optimal levels for N(0,1) ---
def compute_nf4_levels():
    """Compute the 16 NF4 quantization levels: centroids of equal-probability bins of N(0,1)."""
    n_levels = 16
    # Equal-probability bin edges via inverse CDF
    edges = stats.norm.ppf(np.linspace(0, 1, n_levels + 1))
    edges[0], edges[-1] = -np.inf, np.inf  # first/last bins extend to +/-inf

    # Centroid of each bin: E[X | X in [a, b]] for X ~ N(0,1)
    levels = []
    for k in range(n_levels):
        a, b = edges[k], edges[k+1]
        # E[X | a < X < b] = (phi(a) - phi(b)) / (Phi(b) - Phi(a))
        prob = stats.norm.cdf(b) - stats.norm.cdf(a)
        centroid = (stats.norm.pdf(a) - stats.norm.pdf(b)) / prob
        levels.append(centroid)
    return np.array(levels), edges

nf4_levels, nf4_edges = compute_nf4_levels()
print("NF4 quantization levels (optimal for Gaussian weights):")
print(f"  {np.round(nf4_levels, 4).tolist()}")
print(f"  Note: levels cluster near 0 (where Gaussian density is highest)")
print(f"  Spacing near 0: {nf4_levels[8] - nf4_levels[7]:.4f}")
print(f"  Spacing in tail: {nf4_levels[15] - nf4_levels[14]:.4f}")
print(f"  Tail spacing is {(nf4_levels[15]-nf4_levels[14])/(nf4_levels[8]-nf4_levels[7]):.1f}x wider")

# Quantize using NF4
def quantize_nf4(w, levels):
    """Quantize w to nearest NF4 level (after normalizing to unit variance)."""
    scale = w.std().item()
    w_norm = (w / scale).numpy()
    levels_arr = np.array(levels)
    # Nearest-level assignment
    indices = np.argmin(np.abs(w_norm[:, None] - levels_arr[None, :]), axis=1)
    w_hat = torch.tensor(levels_arr[indices], dtype=w.dtype) * scale
    return w_hat

w_nf4_demo = torch.randn(1000) * 0.02
w_hat_nf4 = quantize_nf4(w_nf4_demo, nf4_levels)
w_hat_int4, _, _ = quantize_symmetric(w_nf4_demo, 4)
print(f"\nMSE comparison on Gaussian weights (n=1000):")
print(f"  Uniform INT4:  {((w_nf4_demo - w_hat_int4)**2).mean().item():.8f}")
print(f"  NF4 (optimal): {((w_nf4_demo - w_hat_nf4)**2).mean().item():.8f}")
print(f"  NF4 wins because it allocates more levels near 0 where the Gaussian density peaks.")

## 🤖 Why This Matters for AI

Quantization is not optional knowledge for anyone deploying LLMs — it is *the* technique that
makes large-model inference feasible on real hardware.

**Deployment stacks — all quantization-based:**

- **llama.cpp** (the tool running LLMs on laptops/phones) uses GGUF quantization formats
  (Q4_K_M, Q5_K_S, etc.) — group quantization with importance-weighted bit allocation.
- **vLLM** and **TensorRT-LLM** use AWQ/GPTQ for production LLM serving at scale.
- **bitsandbytes** uses NF4 for QLoRA — the most popular method for fine-tuning LLMs on
  consumer hardware.

**The fundamental deployment equation:** "Can this model run on my GPU?" is a quantization
question:

| Model | float16 | INT8 | INT4 |
| :--- | :--- | :--- | :--- |
| **7B params** | 14 GB | 7 GB | 3.5 GB |
| **13B params** | 26 GB | 13 GB | 6.5 GB |
| **70B params** | 140 GB | 70 GB | 35 GB |

The memory formula: $\text{Memory (GB)} = \frac{\text{params} \times \text{bits per param}}{8 \times 10^9}$ (plus a small overhead for scales in group quantization).

**A 70B model in float16 needs 140 GB (multiple A100s); in INT4 it fits in 35 GB (one A100,
or two consumer GPUs).** This is the difference between a $50k server and a $2k desktop.

**Speed:** INT4/INT8 matrix multiplications are faster than float16 on modern GPUs
(NVIDIA's INT8 Tensor Cores achieve $2\times$ the throughput of float16), making
quantization a win for both memory *and* latency.

Below we quantize a small model end-to-end and measure the actual output degradation,
confirming the theory's predictions: INT8 is near-lossless, INT4 with proper calibration
is surprisingly good, and naive RTN at INT4 is noticeably worse.

In [ ]:
# --- Quantize a small model to INT8/INT4, measure output MSE ---
torch.manual_seed(42)

# Build a small 3-layer MLP (stand-in for a transformer block)
model = nn.Sequential(
    nn.Linear(64, 128), nn.GELU(),
    nn.Linear(128, 128), nn.GELU(),
    nn.Linear(128, 64)
)

# Reference output in full precision
x_test = torch.randn(32, 64)
with torch.no_grad():
    y_ref = model(x_test)

def quantize_model_weights(model, bits, group_size=None):
    """Quantize all weight matrices in a model (leave biases in full precision)."""
    import copy
    state = model.state_dict()
    for name, param in state.items():
        if 'weight' in name:
            if group_size is not None:
                state[name], _ = quantize_group(param, bits=bits, group_size=group_size)
            else:
                state[name], _, _ = quantize_symmetric(param, bits=bits)
    model_q = copy.deepcopy(model)
    model_q.load_state_dict(state)
    return model_q

print("Output MSE vs full-precision reference (lower is better):")
print("=" * 60)
configs = [
    ("INT8 (per-tensor)",     8, None),
    ("INT4 (per-tensor RTN)", 4, None),
    ("INT4 (group=128)",      4, 128),
    ("INT4 (group=32)",       4, 32),
    ("INT3 (per-tensor RTN)", 3, None),
    ("INT2 (per-tensor RTN)", 2, None),
]
results = []
for label, bits, gs in configs:
    model_q = quantize_model_weights(model, bits, gs)
    with torch.no_grad():
        y_q = model_q(x_test)
    mse = ((y_ref - y_q)**2).mean().item()
    cos_sim = F.cosine_similarity(y_ref.flatten().unsqueeze(0),
                                   y_q.flatten().unsqueeze(0)).item()
    results.append((label, mse, cos_sim))
    print(f"  {label:25s}  MSE={mse:.6f}  cos_sim={cos_sim:.6f}")

print(f"\nKey observations:")
print(f"  * INT8 is near-lossless (MSE ~ {results[0][1]:.6f})")
print(f"  * INT4+group quantization dramatically improves over naive RTN")
print(f"  * Below 4 bits, quality degrades rapidly (the 'cliff')")

# --- GPTQ-style sequential quantization for a single linear layer ---
print("\n" + "=" * 60)
print("GPTQ-style sequential quantization (single layer)")
print("=" * 60)

def gptq_quantize_layer(W, X_calib, bits=4):
    """GPTQ-style sequential quantization: quantize one column at a time,
    redistributing error to remaining columns using the inverse Hessian."""
    W = W.clone()
    n_out, n_in = W.shape
    qmax = 2**(bits - 1) - 1

    # Hessian approximation: H = X^T X (+ damping for numerical stability)
    H = X_calib.T @ X_calib / X_calib.shape[0]
    H += 1e-4 * torch.eye(n_in) * H.diag().mean()  # damping
    H_inv = torch.linalg.inv(H)

    for j in range(n_in):
        # Current column
        w_col = W[:, j]

        # Quantize this column
        scale_col = w_col.abs().max() / qmax
        scale_col = max(scale_col.item(), 1e-10)
        q_col = torch.clamp(torch.round(w_col / scale_col), -qmax, qmax)
        w_hat_col = q_col * scale_col

        # Quantization error
        delta = w_col - w_hat_col

        # Update remaining columns to compensate (GPTQ's key idea)
        W[:, j] = w_hat_col
        if j < n_in - 1:
            W[:, j+1:] += delta.unsqueeze(1) * H_inv[j, j+1:].unsqueeze(0) / H_inv[j, j]

    return W

# Compare RTN vs GPTQ on one layer
torch.manual_seed(42)
layer = model[0]  # first linear layer
W_orig = layer.weight.data.clone()
X_calib = torch.randn(256, 64)  # calibration data

# RTN baseline
W_rtn, _, _ = quantize_symmetric(W_orig, 4)
# GPTQ
W_gptq = gptq_quantize_layer(W_orig, X_calib, bits=4)

# Measure output MSE on calibration data
y_orig = X_calib @ W_orig.T
y_rtn = X_calib @ W_rtn.T
y_gptq = X_calib @ W_gptq.T

mse_rtn = ((y_orig - y_rtn)**2).mean().item()
mse_gptq = ((y_orig - y_gptq)**2).mean().item()
print(f"\n  Layer output MSE (INT4):")
print(f"    RTN  (naive):      {mse_rtn:.6f}")
print(f"    GPTQ (Hessian):    {mse_gptq:.6f}")
if mse_gptq > 0:
    print(f"    GPTQ improvement:  {mse_rtn/mse_gptq:.1f}x lower MSE")
else:
    print(f"    GPTQ improvement:  significant")
print(f"\n  GPTQ redistributes each column's rounding error to later columns")
print(f"  using the inverse Hessian -- the same weights that matter most for")
print(f"  the output (high curvature in H) get the most careful treatment.")

## 🏋️ Exercises

Six levels, in order: mechanical computation → conceptual understanding → derivation →
implementation → application → open-ended challenge. Solutions/outlines are in the collapsed
`Solution` blocks below each exercise — try before revealing.

### 🔢 COMPUTE
1. Quantize the vector $[0.23, -1.05, 0.78, -0.12, 2.41]$ to INT8 with symmetric
   quantization. Compute the scale, the integer values, and the reconstruction error for
   each element.

<details>
<summary>💡 Solution</summary>

Scale $s = \max(|w|) / (2^7 - 1) = 2.41 / 127 \approx 0.01898$. Integer values:
$q = \text{round}(w / s) = [12, -55, 41, -6, 127]$. Reconstructed:
$\hat{w} = q \cdot s \approx [0.2278, -1.0441, 0.7782, -0.1139, 2.4106]$.
Per-element errors: $|w - \hat{w}| \approx [0.0022, 0.0059, 0.0018, 0.0061, 0.0006]$.
All errors are $\leq s/2 \approx 0.0095$, confirming the worst-case bound from Theory §2.

</details>

### 💭 UNDERSTAND
2. Why does per-channel quantization outperform per-tensor quantization? Give a concrete
   example with two channels whose scales differ by $10\times$.

<details>
<summary>💡 Solution</summary>

Consider two output channels: channel A with weights in $[-0.05, 0.05]$ and channel B with
weights in $[-0.5, 0.5]$. Per-tensor quantization uses the global range $[-0.5, 0.5]$, giving
step size $s = 1.0/15 \approx 0.067$ for INT4. Channel A's weights (range 0.1) get only
$\sim\!1$–$2$ distinct quantization levels — nearly all information is destroyed. Per-channel
quantization gives channel A its own scale $s_A = 0.1/15 \approx 0.0067$, preserving $15\times$
more resolution. The fundamental issue: one scale can't serve channels with very different
dynamic ranges — the wide channel dominates and the narrow channel loses all its precision.

</details>

### ✏️ DERIVE
3. For a weight $w \sim \mathcal{N}(0, \sigma^2)$, derive the MSE-optimal quantization levels
   for $b = 2$ bits (4 levels). Show that they are *not* equally spaced — the optimal levels
   cluster near 0 where the density is highest.

<details>
<summary>💡 Solution outline</summary>

For 4 levels and a symmetric distribution, we need 2 positive levels $q_1, q_2$ (and their
negatives $-q_1, -q_2$) with a decision boundary at 0 and at some threshold $t > 0$. The
MSE-optimal assignment (Lloyd-Max quantizer) sets each level to the centroid of its Voronoi
region: $q_1 = \mathbb{E}[|X| \mid |X| < t]$
and $q_2 = \mathbb{E}[|X| \mid |X| \geq t]$. Iterating between updating levels and boundaries
(the Lloyd algorithm) converges to $q_1 \approx 0.4528\sigma$, $q_2 \approx 1.5104\sigma$,
$t \approx 0.9816\sigma$. The spacing is: $|q_1 - 0| = 0.45\sigma$ but
$|q_2 - q_1| = 1.06\sigma$ — the outer level is $2.3\times$ farther from its neighbor,
confirming that optimal levels cluster near 0 (high density) and spread out in the tails
(low density). This is exactly the NF4 principle from Theory §5.

</details>

### 🐍 IMPLEMENT
4. Implement GPTQ-style sequential quantization for a single linear layer (weight matrix
   $\mathbf{W} \in \mathbb{R}^{m \times n}$). Compare the output MSE
   $\|\mathbf{W}\mathbf{x} - \hat{\mathbf{W}}\mathbf{x}\|^2$ to naive round-to-nearest on
   calibration data.

<details>
<summary>✅ How to know you're right</summary>

GPTQ should achieve at least $2$–$5\times$ lower output MSE than RTN at INT4 on a
random linear layer with Gaussian calibration data. The improvement comes from error
compensation: each column's rounding error is redistributed to later columns using the
inverse Hessian. Verify by running the implementation in Cell 8 above — it demonstrates
exactly this comparison.

</details>

### 🤖 APPLY
5. Quantize a small GPT-2-style model to INT8 and INT4 (group size 128). Measure perplexity
   on 1000 tokens. At what bit-width does quality degrade noticeably?

<details>
<summary>✅ What you should observe</summary>

INT8 perplexity should be within $1\%$ of float16. INT4 with group quantization ($g=128$)
should be within $1$–$3\%$. INT4 per-tensor (no groups) will show $5$–$15\%$ degradation.
INT3 per-tensor will show $30$+$\%$ degradation — the cliff. You can run this with
HuggingFace's `transformers` library: load `gpt2`, quantize its `.weight` tensors using the
functions from Cell 6, and evaluate with `lm_eval` or a simple next-token-prediction loop.
The exact numbers depend on the evaluation data, but the pattern (gentle slope → cliff) is
robust.

</details>

### 🚀 CHALLENGE
6. AWQ protects "salient" channels by scaling them up before quantization. Derive the
   optimal per-channel scale $s_j$ that minimizes $\mathbb{E}[\|\mathbf{W}\mathbf{x} -
   \hat{\mathbf{W}}\mathbf{x}\|^2]$ given activation statistics $\mathbb{E}[x_j^2]$ and a
   model of uniform quantization noise with variance $\propto (\max_i |W_{ij}|\, s_j)^2$.

<details>
<summary>🔍 What a strong answer covers</summary>

Model the quantization error on the rescaled weight $W'_{ij} = W_{ij} \cdot s_j$ as uniform
noise with variance $\propto (\max_i |W_{ij}| \cdot s_j)^2 / (2^{2b})$. The output error
contributed by channel $j$ is this times $\mathbb{E}[x_j^2] / s_j^2$ (since we divide by
$s_j$ after dequantization). The total expected output error is
$\sum_j \frac{\mathbb{E}[x_j^2]}{s_j^2} \cdot \frac{(\max_i |W_{ij}| \cdot s_j)^2}{2^{2b}}$.
Taking the derivative with respect to $s_j$ and setting to zero, the optimal scale balances
the quantization resolution gained by scaling up against the amplified quantization step
size. For the simple uniform-noise model, this yields $s_j^* \propto
(\mathbb{E}[x_j^2])^{1/2}$ — channels with larger activations get a larger protective
scale, which is exactly AWQ's heuristic. A more careful derivation (accounting for the
clipping boundary) gives a slightly different exponent, but the core insight is the same:
protect the channels that the model *uses* the most.

</details>

---

## 📚 Further Reading
- Frantar et al., ["GPTQ: Accurate Post-Training Quantization for Generative Pre-trained Transformers"](https://arxiv.org/abs/2210.17323) (2022)
- Lin et al., ["AWQ: Activation-aware Weight Quantization for LLM Compression and Acceleration"](https://arxiv.org/abs/2306.00978) (2023)
- Dettmers et al., ["QLoRA: Efficient Finetuning of Quantized LLMs"](https://arxiv.org/abs/2305.14314) (2023)
- Dettmers et al., ["LLM.int8(): 8-bit Matrix Multiplication for Transformers at Scale"](https://arxiv.org/abs/2208.07339) (2022)
- Nagel et al., ["A White Paper on Neural Network Quantization"](https://arxiv.org/abs/2106.08295) (2021)

---

*End of Module 06 — Numerical Methods*